In [3]:
# ============================================================
# r_cpu_apollo_lcm.ipynb
# Apollo CPU benchmark for the windmill latent class model
#
# This notebook estimates a two-class latent class model (LCM)
# for a windmill discrete-choice experiment using Apollo on CPU.
# The code is written as a package-native Apollo benchmark and
# records repeated multi-start estimations for later comparison
# with accelerated likelihood implementations.
#
# Scope:
# - Apollo-native CPU estimation of a two-class latent class model.
# - Class-specific MNL utilities for the windmill choice experiment.
# - Individual-specific class allocation using age, gender, and education.
# - A single Apollo run using the deterministic reference start.
# - Reproducible multi-start estimation with random starting values.
# - Runtime, convergence, log-likelihood, and coefficient records.
# - Hessian and post-estimation standard errors are omitted to focus on
#   estimation-time benchmarking and local-optimum screening.
#
# Main outputs:
# - results/raw/apollo_lcm_cpu_single_run.csv
# - results/raw/apollo_lcm_cpu_single_run_coef.csv
# - results/raw/apollo_lcm_cpu_run_*.csv
# - results/raw/apollo_lcm_cpu_all_runs.csv
# - results/benchmark_summary_apollo_lcm_cpu.csv
# - results/session_info_apollo_lcm_cpu.txt
# ============================================================


In [4]:
# -------------------------------------------------------------------
# 1. Packages
# -------------------------------------------------------------------
# Apollo is used as the package-native CPU estimator for the LCM.
# The Posit Package Manager is used in Colab to speed up installation
# and to improve reproducibility of binary package resolution.

system("apt-get update -qq")
system("apt-get install -y build-essential cmake pkg-config libcurl4-openssl-dev libssl-dev libxml2-dev")

options(HTTPUserAgent = sprintf(
  "R/%s R (%s)",
  getRversion(),
  paste(
    getRversion(),
    R.version["platform"],
    R.version["arch"],
    R.version["os"]
  )
))

ubuntu_codename <- tryCatch(
  system("bash -lc '. /etc/os-release; echo ${VERSION_CODENAME}'", intern = TRUE),
  error = function(e) "jammy"
)

ppm_repo <- sprintf(
  "https://packagemanager.posit.co/cran/__linux__/%s/latest",
  ubuntu_codename[1]
)

options(repos = c(CRAN = ppm_repo))

cat("Using CRAN repo:\n", getOption("repos"), "\n\n")

install.packages("apollo", Ncpus = parallel::detectCores())

suppressPackageStartupMessages({
  library(apollo)
})

pkg_versions <- data.frame(
  package = "apollo",
  version = as.character(packageVersion("apollo"))
)

print(pkg_versions)


Using CRAN repo:
 https://packagemanager.posit.co/cran/__linux__/jammy/latest 



Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘listenv’, ‘parallelly’, ‘zoo’, ‘future’, ‘globals’, ‘SparseM’, ‘MatrixModels’, ‘miscTools’, ‘sandwich’, ‘rngWELL’, ‘truncnorm’, ‘future.apply’, ‘mcmc’, ‘quantreg’, ‘maxLik’, ‘mnormt’, ‘mvtnorm’, ‘randtoolbox’, ‘numDeriv’, ‘Deriv’, ‘matrixStats’, ‘coda’, ‘bgw’, ‘Rsolnp’, ‘MCMCpack’, ‘RcppArmadillo’, ‘RcppEigen’




  package version
1  apollo   0.3.9


In [5]:
# -------------------------------------------------------------------
# 2. Project paths and benchmark settings
# -------------------------------------------------------------------
# All outputs are written below the current project root. The naming
# convention separates run-level raw results from the compact summary
# table used in the manuscript or supplementary benchmark archive.

proj_root <- getwd()

raw_results_dir <- file.path(proj_root, "results", "raw")
summary_file <- file.path(proj_root, "results", "benchmark_summary_apollo_lcm_cpu.csv")
session_info_file <- file.path(proj_root, "results", "session_info_apollo_lcm_cpu.txt")

dir.create(raw_results_dir, recursive = TRUE, showWarnings = FALSE)

# Benchmark metadata.

package <- "Apollo"
implementation <- "apollo_lcm_cpu"
backend <- "CPU"
optimiser <- "Apollo_bgw"

base_seed <- 1234L
n_runs <- 500L
draw_sizes <- "not used in LCM"

# Data are loaded here so that the benchmark header can report sample size.

url <- "https://raw.githubusercontent.com/edsandorf/evdce/main/Data/data-windmills.csv"
database <- read.csv(url, header = TRUE)
database <- database[!is.na(database$choice), ]

database$educ_2 <- ifelse(database$education == 2, 1, 0)
database$educ_3 <- ifelse(database$education == 3, 1, 0)

n_individuals <- length(unique(database$id_individual))
n_observations <- nrow(database)

tasks_tab <- table(database$id_individual)
tasks_person <- if (length(unique(tasks_tab)) == 1) {
  as.character(unique(tasks_tab))
} else {
  "unbalanced"
}

models <- vector("list", n_runs)
run_results <- vector("list", n_runs)

# Save session and hardware information for reproducibility.

cpu_info <- tryCatch(
  system("lscpu", intern = TRUE),
  error = function(e) paste("lscpu failed:", conditionMessage(e))
)

sink(session_info_file)

cat("Benchmark settings\n")
cat("Package      :", package, "\n")
cat("Implementation:", implementation, "\n")
cat("Optimiser    :", optimiser, "\n")
cat("Draw sizes   :", draw_sizes, "\n")
cat("Seed         :", base_seed, "\n")
cat("Reps         :", n_runs, "\n")
cat("Device       :", tolower(backend), "\n")
cat("Individuals  :", n_individuals, "\n")
cat("Tasks/person :", tasks_person, "\n")
cat("Observations :", n_observations, "\n")
cat("Raw output   :", raw_results_dir, "\n")
cat("Summary file :", summary_file, "\n")

cat("\nR session information\n")
print(sessionInfo())

cat("\nPackage versions\n")
cat("apollo: ", as.character(packageVersion("apollo")), "\n", sep = "")

cat("\nCPU information\n")
cat(paste(cpu_info, collapse = "\n"), "\n")

sink()

cat("\nBenchmark settings\n")
cat("Package      :", package, "\n")
cat("Implementation:", implementation, "\n")
cat("Optimiser    :", optimiser, "\n")
cat("Draw sizes   :", draw_sizes, "\n")
cat("Seed         :", base_seed, "\n")
cat("Reps         :", n_runs, "\n")
cat("Device       :", tolower(backend), "\n")
cat("Individuals  :", n_individuals, "\n")
cat("Tasks/person :", tasks_person, "\n")
cat("Observations :", n_observations, "\n")
cat("Raw output   :", raw_results_dir, "\n")
cat("Summary file :", summary_file, "\n")

cat("\nCPU information:\n")
cat(paste(cpu_info, collapse = "\n"), "\n")

cat("\nSession and hardware information saved to:", session_info_file, "\n")


Benchmark settings
Package      : Apollo 
Implementation: apollo_lcm_cpu 
Optimiser    : Apollo_bgw 
Draw sizes   : not used in LCM 
Seed         : 1234 
Reps         : 500 
Device       : cpu 
Individuals  : 1000 
Tasks/person : unbalanced 
Observations : 8633 
Raw output   : /content/results/raw 
Summary file : /content/results/benchmark_summary_apollo_lcm_cpu.csv 

CPU information:
Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  8
On-line CPU(s) list:                     0-7
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
CPU family:                              6
Model:                                   79
Thread(s) per core:                      2
Core(s) per socket:              

In [6]:
# ============================================================
# 3. Define the Apollo latent class model
# ============================================================
# This section defines the data, initial parameter vector, class-
# allocation model, and class-specific choice probabilities.
# The behavioural specification is kept explicit so that the Apollo
# version can serve as the transparent reference implementation.

# ------------------------------------------------------------
# 3.1 Apollo control settings and data
# ------------------------------------------------------------
# The model is a two-class LCM. The individual identifier is required
# because each respondent contributes repeated choice observations.

apollo_control <- list(
  modelName  = "LCM_2class_windmills",
  modelDescr = "2-class latent class model for the windmill DCE",
  indivID    = "id_individual",
  nCores     = 8
)



# ------------------------------------------------------------
# 3.2 Starting values and identification normalisation
# ------------------------------------------------------------
# The suffixes _a and _b denote class-specific parameters.
# Class b is the reference class in the allocation model, so the class-b
# allocation constant and covariate effects are fixed to zero.

apollo_beta <- c(
  asc_sq_a = 0.1,
  asc_sq_b = 0.0,

  b_medium_farms_a = 0.1,
  b_medium_farms_b = 0.0,

  b_small_farms_a = 0.1,
  b_small_farms_b = 0.0,

  b_medium_height_a = 0.1,
  b_medium_height_b = 0.0,

  b_low_height_a = 0.1,
  b_low_height_b = 0.0,

  b_redkite_a = -0.1,
  b_redkite_b = -0.1,

  b_distance_a = 0.1,
  b_distance_b = 0.1,

  b_cost_a = -0.1,
  b_cost_b = -0.1,

  delta_a = 0.1,
  gamma_age_a = 0,
  gamma_female_a = 0,
  gamma_educ2_a = 0,
  gamma_educ3_a = 0,

  delta_b = 0,
  gamma_age_b = 0,
  gamma_female_b = 0,
  gamma_educ2_b = 0,
  gamma_educ3_b = 0
)

apollo_fixed <- c(
  "delta_b",
  "gamma_age_b",
  "gamma_female_b",
  "gamma_educ2_b",
  "gamma_educ3_b"
)

# Keep a clean copy of the deterministic reference starting values.
# The multi-start cell later overwrites apollo_beta inside the loop,
# so this object makes the single-run cell independent and reproducible.

apollo_beta_base <- apollo_beta

# ------------------------------------------------------------
# 3.3 Latent class parameters and class allocation
# ------------------------------------------------------------
# Apollo expects class-specific preference parameters as lists. The
# class-allocation probabilities are generated from a multinomial logit
# allocation model with respondent characteristics.

apollo_lcPars <- function(apollo_beta, apollo_inputs) {

  lcpars <- list()

  lcpars[["asc_sq"]] <- list(asc_sq_a, asc_sq_b)

  lcpars[["b_medium_farms"]] <- list(
    b_medium_farms_a,
    b_medium_farms_b
  )

  lcpars[["b_small_farms"]] <- list(
    b_small_farms_a,
    b_small_farms_b
  )

  lcpars[["b_medium_height"]] <- list(
    b_medium_height_a,
    b_medium_height_b
  )

  lcpars[["b_low_height"]] <- list(
    b_low_height_a,
    b_low_height_b
  )

  lcpars[["b_redkite"]] <- list(
    b_redkite_a,
    b_redkite_b
  )

  lcpars[["b_distance"]] <- list(
    b_distance_a,
    b_distance_b
  )

  lcpars[["b_cost"]] <- list(
    b_cost_a,
    b_cost_b
  )

  V <- list()

  V[["class_a"]] <-
    delta_a +
    gamma_age_a    * age +
    gamma_female_a * female +
    gamma_educ2_a  * educ_2 +
    gamma_educ3_a  * educ_3

  V[["class_b"]] <-
    delta_b +
    gamma_age_b    * age +
    gamma_female_b * female +
    gamma_educ2_b  * educ_2 +
    gamma_educ3_b  * educ_3

  classAlloc_settings <- list(
    classes   = c(class_a = 1, class_b = 2),
    utilities = V
  )

  lcpars[["pi_values"]] <- apollo_classAlloc(classAlloc_settings)

  return(lcpars)
}

apollo_inputs <- apollo_validateInputs()

# ------------------------------------------------------------
# 3.4 Choice probability function
# ------------------------------------------------------------
# For each class, the model first computes an MNL probability for each
# choice task. The task probabilities are then multiplied within each
# respondent using apollo_panelProd(). Finally, apollo_lc() combines the
# class-specific probabilities using the class-allocation probabilities.

apollo_probabilities <- function(apollo_beta, apollo_inputs, functionality = "estimate") {

  apollo_attach(apollo_beta, apollo_inputs)
  on.exit(apollo_detach(apollo_beta, apollo_inputs))

  P <- list()

  mnl_settings <- list(
    alternatives = c(alt1 = 1, alt2 = 2, alt3 = 3),
    avail        = list(alt1 = 1, alt2 = 1, alt3 = 1),
    choiceVar    = choice
  )

  for (s in 1:2) {

    V <- list()

    # Alternative 1 is the status-quo alternative and is represented
    # by a class-specific status-quo constant.

    V[["alt1"]] <-
      asc_sq[[s]] * alt1_sq

    # Alternatives 2 and 3 are described by windmill attributes.
    # Coefficients are class-specific and indexed by s.

    V[["alt2"]] <-
      b_medium_farms[[s]]  * alt2_farm2 +
      b_small_farms[[s]]   * alt2_farm3 +
      b_medium_height[[s]] * alt2_height2 +
      b_low_height[[s]]    * alt2_height3 +
      b_redkite[[s]]       * alt2_redkite +
      b_distance[[s]]      * alt2_distance +
      b_cost[[s]]          * alt2_cost

    V[["alt3"]] <-
      b_medium_farms[[s]]  * alt3_farm2 +
      b_small_farms[[s]]   * alt3_farm3 +
      b_medium_height[[s]] * alt3_height2 +
      b_low_height[[s]]    * alt3_height3 +
      b_redkite[[s]]       * alt3_redkite +
      b_distance[[s]]      * alt3_distance +
      b_cost[[s]]          * alt3_cost

    mnl_settings$utilities <- V

    P[[paste0("Class_", s)]] <-
      apollo_mnl(mnl_settings, functionality)

    P[[paste0("Class_", s)]] <-
      apollo_panelProd(
        P[[paste0("Class_", s)]],
        apollo_inputs,
        functionality
      )
  }

  lc_settings <- list(
    inClassProb = P,
    classProb   = pi_values
  )

  P[["model"]] <- apollo_lc(lc_settings, apollo_inputs, functionality)

  P <- apollo_prepareProb(P, apollo_inputs, functionality)

  return(P)
}


Several observations per individual detected based on the value of
  id_individual. Setting panelData in apollo_control set to TRUE.
All checks on apollo_control completed.
All checks on database completed.


In [7]:
# ============================================================
# 4. Single-run estimation
# ============================================================
# This cell estimates the two-class LCM once using the deterministic
# reference starting values defined in Section 3. It is useful for a
# quick model check, for inspecting the Apollo output, and for producing
# a transparent single-run baseline before launching the multi-start
# benchmark.

apollo_beta_single <- apollo_beta_base

# Initial log-likelihood at the deterministic starting point.

p0_single <- apollo_probabilities(
  apollo_beta_single,
  apollo_inputs,
  functionality = "estimate"
)

initial_loglik_single <- sum(log(as.numeric(unlist(p0_single))))

# The Hessian is disabled here to keep the single run consistent with
# the runtime benchmark. Standard errors can be restored by removing
# hessianRoutine = "none" when the final behavioural model is reported.

model_single <- apollo_estimate(
  apollo_beta_single,
  apollo_fixed,
  apollo_probabilities,
  apollo_inputs,
  estimate_settings = list(
    hessianRoutine = "none",
    maxIterations = 5000,
    printLevel = 0,
    writeIter = FALSE
  )
)

estimation_time_single <- model_single$timeEst
final_loglik_single <- model_single$maximum
bgw_iterations_single <- model_single$iterations
convergence_detail_single <- model_single$message

# Combine estimated free parameters with fixed parameters so that the
# output contains the complete parameter vector used in the LCM.

estimates_single <- apollo_beta_single
free_par_single <- setdiff(names(apollo_beta_single), apollo_fixed)
estimates_single[free_par_single] <- model_single$estimate[free_par_single]

single_coef_result <- data.frame(
  package = package,
  implementation = implementation,
  backend = backend,
  optimiser = optimiser,
  seed = NA_integer_,
  run = 1L,
  parameter = names(apollo_beta_single),
  initial_value = as.numeric(apollo_beta_single),
  estimate = as.numeric(estimates_single),
  fixed = names(apollo_beta_single) %in% apollo_fixed
)

write.csv(
  single_coef_result,
  file.path(raw_results_dir, "apollo_lcm_cpu_single_run_coef.csv"),
  row.names = FALSE
)

single_run_result <- data.frame(
  package = package,
  implementation = implementation,
  backend = backend,
  optimiser = optimiser,
  seed = NA_integer_,
  run = 1L,
  initial_loglik = initial_loglik_single,
  estimation_time_sec = estimation_time_single,
  final_loglik = final_loglik_single,
  bgw_iterations = bgw_iterations_single,
  convergence_detail = convergence_detail_single
)

write.csv(
  single_run_result,
  file.path(raw_results_dir, "apollo_lcm_cpu_single_run.csv"),
  row.names = FALSE
)

model <- model_single

print(single_run_result)
apollo_modelOutput(model_single)


Preparing user-defined functions.

Testing likelihood function...
Apollo found a model component of type classAlloc without a
  componentName. The name was set to "classAlloc" by default.
INFORMATION: Setting "avail" is missing, so full availability is
  assumed. 
Apollo found a model component of type MNL without a componentName. The
  name was set to "Class_1" by default.

Overview of choices for MNL model component Class_1:
                                    alt1    alt2    alt3
Times available                  8633.00 8633.00 8633.00
Times chosen                     4850.00 1885.00 1898.00
Percentage chosen overall          56.18   21.83   21.99
Percentage chosen when available   56.18   21.83   21.99

Apollo found a model component of type MNL without a componentName. The
  name was set to "Class_2" by default.

Overview of choices for MNL model component Class_2:
                                    alt1    alt2    alt3
Times available                  8633.00 8633.00 8633.00
Tim

In [11]:
# ============================================================
# 5. Multi-start estimation
# ============================================================
# Latent class models can be sensitive to local optima. This benchmark
# therefore estimates the same two-class model repeatedly using
# reproducible random starting values. Each run draws all free starting
# values from U(-1, 1), while the class-b allocation parameters remain
# fixed to impose the identification normalisation.

for (i in seq_len(n_runs)) {

  seed_i <- base_seed + i - 1L
  set.seed(seed_i)

  apollo_beta <- c(
    asc_sq_a = runif(1, -1, 1),
    asc_sq_b = runif(1, -1, 1),

    b_medium_farms_a = runif(1, -1, 1),
    b_medium_farms_b = runif(1, -1, 1),

    b_small_farms_a = runif(1, -1, 1),
    b_small_farms_b = runif(1, -1, 1),

    b_medium_height_a = runif(1, -1, 1),
    b_medium_height_b = runif(1, -1, 1),

    b_low_height_a = runif(1, -1, 1),
    b_low_height_b = runif(1, -1, 1),

    b_redkite_a = runif(1, -1, 1),
    b_redkite_b = runif(1, -1, 1),

    b_distance_a = runif(1, -1, 1),
    b_distance_b = runif(1, -1, 1),

    b_cost_a = runif(1, -1, 1),
    b_cost_b = runif(1, -1, 1),

    delta_a = runif(1, -1, 1),
    gamma_age_a = runif(1, -1, 1),
    gamma_female_a = runif(1, -1, 1),
    gamma_educ2_a = runif(1, -1, 1),
    gamma_educ3_a = runif(1, -1, 1),

    delta_b = 0,
    gamma_age_b = 0,
    gamma_female_b = 0,
    gamma_educ2_b = 0,
    gamma_educ3_b = 0
  )

  # Initial log-likelihood at the random starting point. This is saved
  # to diagnose whether poor final solutions are associated with extreme
  # or unstable starting values.

  p0 <- apollo_probabilities(
    apollo_beta,
    apollo_inputs,
    functionality = "estimate"
  )

  initial_loglik <- sum(log(as.numeric(unlist(p0))))

  # Standard errors and post-estimation Hessian calculations are disabled
  # because this notebook is designed for repeated runtime benchmarking
  # and local-optimum screening.

  model_i <- apollo_estimate(
    apollo_beta,
    apollo_fixed,
    apollo_probabilities,
    apollo_inputs,
    estimate_settings = list(
      hessianRoutine = "none",
      maxIterations = 5000,
      printLevel = 0,
      writeIter = FALSE
    )
  )

  estimation_time_sec <- model_i$timeEst
  final_loglik <- model_i$maximum
  bgw_iterations <- model_i$iterations
  convergence_detail <- model_i$message

  models[[i]] <- model_i

  # Combine estimated free parameters with fixed parameters so that each
  # run-level coefficient file has one row for every model parameter.

  estimates <- apollo_beta
  free_par <- setdiff(names(apollo_beta), apollo_fixed)
  estimates[free_par] <- model_i$estimate[free_par]

  coef_result <- data.frame(
    package = package,
    implementation = implementation,
    backend = backend,
    optimiser = optimiser,
    seed = seed_i,
    run = i,
    parameter = names(apollo_beta),
    initial_value = as.numeric(apollo_beta),
    estimate = as.numeric(estimates),
    fixed = names(apollo_beta) %in% apollo_fixed
  )

  write.csv(
    coef_result,
    file.path(
      raw_results_dir,
      paste0("apollo_lcm_cpu_run_", sprintf("%03d", i), "_seed_", seed_i, ".csv")
    ),
    row.names = FALSE
  )

  run_results[[i]] <- data.frame(
    package = package,
    implementation = implementation,
    backend = backend,
    optimiser = optimiser,
    seed = seed_i,
    run = i,
    initial_loglik = initial_loglik,
    estimation_time_sec = estimation_time_sec,
    final_loglik = final_loglik,
    bgw_iterations = bgw_iterations,
    convergence_detail = convergence_detail
  )

  cat("run =", i,
      "seed =", seed_i,
      "LL =", final_loglik,
      "time =", estimation_time_sec,
      "\n")
}


Preparing user-defined functions.

Testing likelihood function...
Apollo found a model component of type classAlloc without a
  componentName. The name was set to "classAlloc" by default.
INFORMATION: Setting "avail" is missing, so full availability is
  assumed. 
Apollo found a model component of type MNL without a componentName. The
  name was set to "Class_1" by default.

Overview of choices for MNL model component Class_1:
                                    alt1    alt2    alt3
Times available                  8633.00 8633.00 8633.00
Times chosen                     4850.00 1885.00 1898.00
Percentage chosen overall          56.18   21.83   21.99
Percentage chosen when available   56.18   21.83   21.99

Apollo found a model component of type MNL without a componentName. The
  name was set to "Class_2" by default.

Overview of choices for MNL model component Class_2:
                                    alt1    alt2    alt3
Times available                  8633.00 8633.00 8633.00
Tim

ERROR: Error in run_results[[i]] <- data.frame(package = package, implementation = implementation, : more elements supplied than there are to replace


In [14]:
# ============================================================
# 6. Aggregate run-level results
# ============================================================

required_cols <- c(
  "package",
  "implementation",
  "backend",
  "optimiser",
  "seed",
  "run",
  "initial_loglik",
  "estimation_time_sec",
  "final_loglik",
  "bgw_iterations",
  "convergence_detail"
)


# ------------------------------------------------------------
# 6.1 Safely combine / recover run-level results
# ------------------------------------------------------------

# Case 1: original list from the multi-start loop
if (is.list(run_results) && !is.data.frame(run_results)) {

  run_results <- do.call(
    rbind,
    run_results
  )

# Case 2: Section 6 was accidentally run twice and
# run_results became a matrix
} else if (
  is.matrix(run_results) &&
  all(required_cols %in% rownames(run_results))
) {

  message(
    "run_results is a matrix. Recovering data-frame structure..."
  )

  run_results <- as.data.frame(
    t(run_results),
    stringsAsFactors = FALSE
  )

# Case 3: already correctly combined
} else if (is.data.frame(run_results)) {

  message(
    "run_results is already a data frame; skipping rbind."
  )

} else {

  stop(
    "run_results has an unexpected structure. ",
    "Please inspect run_results or rerun Section 5."
  )
}


# ------------------------------------------------------------
# 6.2 Restore numeric column types
# ------------------------------------------------------------

numeric_cols <- c(
  "seed",
  "run",
  "initial_loglik",
  "estimation_time_sec",
  "final_loglik",
  "bgw_iterations"
)

run_results[numeric_cols] <- lapply(
  run_results[numeric_cols],
  as.numeric
)


# ------------------------------------------------------------
# 6.3 Check required columns
# ------------------------------------------------------------

missing_cols <- setdiff(
  required_cols,
  names(run_results)
)

if (length(missing_cols) > 0) {

  stop(
    "Missing columns in run_results: ",
    paste(missing_cols, collapse = ", ")
  )
}


# ------------------------------------------------------------
# 6.4 Save all run-level results
# ------------------------------------------------------------

all_runs_file <- file.path(
  raw_results_dir,
  "apollo_lcm_cpu_all_runs.csv"
)

write.csv(
  run_results,
  all_runs_file,
  row.names = FALSE
)


# ------------------------------------------------------------
# 6.5 Select the best run
# ------------------------------------------------------------

best_id <- which.max(
  run_results$final_loglik
)

best_run <- run_results[
  best_id,
  ,
  drop = FALSE
]


# ------------------------------------------------------------
# 6.6 Create benchmark summary
# ------------------------------------------------------------

benchmark_summary <- data.frame(

  package = best_run$package,
  implementation = best_run$implementation,
  backend = best_run$backend,
  optimiser = best_run$optimiser,

  seed = best_run$seed,
  run = best_run$run,

  initial_loglik = best_run$initial_loglik,

  mean_estimation_time_sec = mean(
    run_results$estimation_time_sec,
    na.rm = TRUE
  ),

  sd_estimation_time_sec = sd(
    run_results$estimation_time_sec,
    na.rm = TRUE
  ),

  min_estimation_time_sec = min(
    run_results$estimation_time_sec,
    na.rm = TRUE
  ),

  max_estimation_time_sec = max(
    run_results$estimation_time_sec,
    na.rm = TRUE
  ),

  total_estimation_time_sec = sum(
    run_results$estimation_time_sec,
    na.rm = TRUE
  ),

  final_loglik = best_run$final_loglik,

  bgw_iterations = best_run$bgw_iterations,

  convergence_detail = best_run$convergence_detail
)


write.csv(
  benchmark_summary,
  summary_file,
  row.names = FALSE
)


# ------------------------------------------------------------
# 6.7 Set best model as active model
# ------------------------------------------------------------

best_model_id <- as.integer(
  best_run$run
)

model <- models[[best_model_id]]


# ------------------------------------------------------------
# 6.8 Print results
# ------------------------------------------------------------

print(run_results)
print(benchmark_summary)

cat("\n")
cat("========================================\n")
cat(" Best multi-start result\n")
cat("========================================\n")

cat(
  "Number of runs:",
  nrow(run_results),
  "\n"
)

cat(
  "Best run:",
  best_run$run,
  "\n"
)

cat(
  "Best seed:",
  best_run$seed,
  "\n"
)

cat(
  "Best final log-likelihood:",
  best_run$final_loglik,
  "\n"
)

cat(
  "Mean estimation time:",
  benchmark_summary$mean_estimation_time_sec,
  "sec\n"
)

cat(
  "Total estimation time:",
  benchmark_summary$total_estimation_time_sec,
  "sec\n"
)

cat(
  "Summary saved to:",
  summary_file,
  "\n"
)


# ------------------------------------------------------------
# 6.9 Compress all benchmark results into ZIP
# ------------------------------------------------------------

results_dir <- dirname(
  raw_results_dir
)

zip_file <- file.path(
  proj_root,
  "apollo_lcm_cpu_results.zip"
)


# Remove previous ZIP
if (file.exists(zip_file)) {

  file.remove(zip_file)
}


# Collect all files inside results directory
files_to_zip <- list.files(
  results_dir,
  recursive = TRUE,
  full.names = FALSE,
  all.files = FALSE
)


if (length(files_to_zip) > 0) {

  old_wd <- getwd()

  tryCatch(
    {

      setwd(results_dir)

      utils::zip(
        zipfile = normalizePath(
          zip_file,
          winslash = "/",
          mustWork = FALSE
        ),
        files = files_to_zip
      )

    },
    finally = {

      setwd(old_wd)
    }
  )

  cat(
    "ZIP saved to:",
    zip_file,
    "\n"
  )

} else {

  warning(
    "No benchmark result files were found for compression."
  )
}

cat("========================================\n")

run_results is a matrix. Recovering data-frame structure...



    package implementation backend  optimiser seed run initial_loglik
1    Apollo apollo_lcm_cpu     CPU Apollo_bgw 1234   1     -28879.375
2    Apollo apollo_lcm_cpu     CPU Apollo_bgw 1235   2     -21997.503
3    Apollo apollo_lcm_cpu     CPU Apollo_bgw 1236   3     -15637.889
4    Apollo apollo_lcm_cpu     CPU Apollo_bgw 1237   4     -15324.601
5    Apollo apollo_lcm_cpu     CPU Apollo_bgw 1238   5     -21522.893
6    Apollo apollo_lcm_cpu     CPU Apollo_bgw 1239   6     -17699.720
7    Apollo apollo_lcm_cpu     CPU Apollo_bgw 1240   7     -21689.906
8    Apollo apollo_lcm_cpu     CPU Apollo_bgw 1241   8     -22746.226
9    Apollo apollo_lcm_cpu     CPU Apollo_bgw 1242   9     -31907.041
10   Apollo apollo_lcm_cpu     CPU Apollo_bgw 1243  10      -9949.732
11   Apollo apollo_lcm_cpu     CPU Apollo_bgw 1244  11     -38105.263
12   Apollo apollo_lcm_cpu     CPU Apollo_bgw 1245  12     -43203.987
13   Apollo apollo_lcm_cpu     CPU Apollo_bgw 1246  13     -13826.550
14   Apollo apollo_l